# COVID-19 Spread and Government Response  
## An explainer notebook comparing Denmark, Germany, and Great Britain

This notebook documents the analytical work behind our final project website. It contains the data preparation, exploratory analysis, methodological choices, visual design reasoning, and interpretation steps that support the final narrative.

Our project investigates two connected questions:

1. To what extent do official COVID-19 case numbers reflect the actual spread of the pandemic?
2. How is government stringency associated with later differences in spread and mortality?

To explore these questions, we use epidemiological and government response data from Google's COVID-19 Open Data and focus on Denmark, Germany, and Great Britain as comparable European cases.

# Motivation

## What is the dataset?

We use two datasets from Google's COVID-19 Open Data:

- **Epidemiology**
- **Oxford Government Response**

The epidemiology dataset contains daily and cumulative COVID-19 indicators such as confirmed cases, deaths, recoveries, and testing.  
The government response dataset contains policy indicators such as school closures, stay-at-home requirements, restrictions on gatherings, and the composite **stringency index**.

## Why did we choose these datasets?

We chose these datasets because they allow us to connect two important aspects of the pandemic:

- the **measured spread** of COVID-19 through official epidemiological indicators
- the **policy response** of governments over time

This combination makes it possible to compare not only how the pandemic developed across countries, but also how governments reacted and whether those reactions appear associated with later outcomes.

## Why did we choose Denmark, Germany, and Great Britain?

We narrowed the analysis to Denmark, Germany, and Great Britain because they are geographically closer and more comparable than a global set of countries. All three are European countries with high institutional capacity, but they still differ in pandemic trajectories, reporting patterns, and policy responses.

## What was our goal for the end user's experience?

Our goal was to create a data story that feels clear, engaging, and approachable for a general audience. The website is designed to guide the reader through the global development of COVID-19 and then into a more focused comparison of Denmark, Germany, and Great Britain. Rather than presenting the analysis as a technical report, we wanted the user experience to feel like an interactive narrative that gradually reveals patterns in the data.

We also wanted readers to be able to explore the data themselves through interactive elements such as maps, line charts, and year-based comparisons. The purpose of the explainer notebook is different: it provides the behind-the-scenes analytical details, including the preprocessing choices, assumptions, methods, and reasoning that support the website's final narrative.

# Dataset Description

## Epidemiology dataset

The epidemiology dataset contains daily observations of COVID-19 development across many locations and geographic levels. Relevant variables include:

- `date`
- `location_key`
- `new_confirmed`
- `new_deceased`
- `new_recovered`
- `new_tested`
- `cumulative_confirmed`
- `cumulative_deceased`
- `cumulative_recovered`
- `cumulative_tested`

This dataset spans from late 2019 to late 2022 and includes country, regional, and local rows.

## Government response dataset

The Oxford Government Response dataset contains policy indicators related to pandemic management. Relevant variables include:

- `date`
- `location_key`
- `school_closing`
- `workplace_closing`
- `cancel_public_events`
- `restrictions_on_gatherings`
- `stay_at_home_requirements`
- `testing_policy`
- `contact_tracing`
- `vaccination_policy`
- `facial_coverings`
- `stringency_index`

This dataset is structurally compatible with the epidemiology dataset because both use `date` and `location_key`, which makes country-level comparison possible.

# Setup

This notebook documents the analytical workflow behind our final project website.  
We begin by importing the required libraries and loading the datasets used in the analysis.

The project is based on two datasets from Google's COVID-19 Open Data:

- **Epidemiology**
- **Oxford Government Response**

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import plotly.express as px
import plotly.graph_objects as go

import pycountry

/Users/katherinakronborg/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Load raw datasets

The code below assumes the notebook is placed in a folder next to a `data` folder containing the original CSV files.

In [4]:
current_dir = Path.cwd()

data_path1 = current_dir.parent / "data" / "epidemiology.csv"
data_path2 = current_dir.parent / "data" / "oxford-government-response.csv"

epi = pd.read_csv(data_path1, parse_dates=["date"], low_memory=False)
gov = pd.read_csv(data_path2, parse_dates=["date"], low_memory=False)

print("Epidemiology shape:", epi.shape)
print("Government response shape:", gov.shape)

Epidemiology shape: (12525825, 10)
Government response shape: (303969, 22)


# Basic Stats: Understanding the Dataset

Before conducting the main analysis, we first inspect the size, structure, and quality of the two datasets.

Our aim in this section is to answer the following questions:

- How large are the datasets?
- What is the date range?
- What geographic levels are present?
- Which variables have substantial missingness?
- Which variables are most reliable for analysis?

The epidemiology dataset is much larger than the government response dataset and includes multiple geographic levels, including countries, regions, and local areas. This is important because summing across all rows would lead to double-counting. Therefore, one of the most important preprocessing choices in the project is to filter both datasets to **country-level rows only**.

Our initial inspection showed that the epidemiology dataset contains **12,525,825 rows**, **10 columns**, spans **2019-12-31 to 2022-12-30**, and includes **20,905 locations**. The government response dataset contains **303,969 rows**, **22 columns**, and uses the same `date` and `location_key` structure, which makes it compatible for later comparison and merging. :contentReference[oaicite:0]{index=0}

In [5]:
print("Epidemiology shape:", epi.shape)
print("Government response shape:", gov.shape)

print("\nEpidemiology date range:", epi["date"].min(), "to", epi["date"].max())
print("Government response date range:", gov["date"].min(), "to", gov["date"].max())

print("\nUnique epidemiology locations:", epi["location_key"].nunique())
print("Unique government response locations:", gov["location_key"].nunique())

Epidemiology shape: (12525825, 10)
Government response shape: (303969, 22)

Epidemiology date range: 2019-12-31 00:00:00 to 2022-12-30 00:00:00
Government response date range: 2020-01-01 00:00:00 to 2022-07-27 00:00:00

Unique epidemiology locations: 20905
Unique government response locations: 355


## Missingness and reliability

We next inspect missing values in order to determine which variables are suitable for cross-country comparison.

This is especially important for COVID-19 data, because some indicators were reported much more consistently than others.

In [6]:
print("Missing values in epidemiology (%):")
print((epi.isna().mean() * 100).round(2).sort_values(ascending=False))

print("\nMissing values in government response (%):")
print((gov.isna().mean() * 100).round(2).sort_values(ascending=False))

Missing values in epidemiology (%):
cumulative_tested       75.95
new_tested              74.50
new_recovered           68.22
cumulative_recovered    68.14
cumulative_deceased      8.39
new_deceased             6.86
cumulative_confirmed     1.59
new_confirmed            0.40
location_key             0.01
date                     0.00
dtype: float64

Missing values in government response (%):
international_support                 51.76
fiscal_measures                       51.72
emergency_investment_in_healthcare    50.29
investment_in_vaccines                23.43
debt_relief                            5.63
income_support                         5.59
contact_tracing                        1.90
international_travel_controls          1.23
testing_policy                         1.18
vaccination_policy                     0.90
stringency_index                       0.78
facial_coverings                       0.77
public_transport_closing               0.70
public_information_campaigns     

## Geographic structure

The `location_key` variable encodes different geographic levels.  
Country-level rows typically contain no underscore, while regional and local rows do.

This matters because the epidemiology dataset contains multiple resolutions at once, and combining them directly would lead to double-counting.

In [7]:
print("Epidemiology location_key underscore counts:")
print(epi["location_key"].astype(str).str.count("_").value_counts().sort_index())

print("\nGovernment response location_key underscore counts:")
print(gov["location_key"].astype(str).str.count("_").value_counts().sort_index())

Epidemiology location_key underscore counts:
location_key
0      227879
1      784558
2    11513388
Name: count, dtype: int64

Government response location_key underscore counts:
location_key
0    172116
1    131853
Name: count, dtype: int64


### Interpretation

These basic statistics confirm that the epidemiology dataset is very large and contains multiple geographic resolutions, while the government response dataset is smaller and more policy-focused. The shared `date` and `location_key` structure makes the two datasets analytically compatible.

The missing-value patterns also suggest that **confirmed cases** and **deaths** are more reliable variables than **recoveries** and **testing**, especially for cross-country comparison. This is one of the reasons why our main analysis emphasizes confirmed cases, deaths, and government stringency rather than recovery counts alone.

# Data Cleaning and Preprocessing

This section describes the preprocessing choices made before the main analysis.

The raw datasets are large and contain multiple geographic levels, reporting corrections, and variables with uneven completeness. The goal of preprocessing is therefore to create a cleaner and more consistent country-level dataset for comparison.

Our main preprocessing steps were:

1. filter to country-level rows only  
2. parse dates consistently  
3. handle negative daily values  
4. smooth noisy daily time series  
5. normalize selected variables by population  
6. prepare country codes for mapping  
7. create a cleaned merged dataset for final analysis

## 1. Country-level filtering

The epidemiology dataset contains country, regional, and local rows.  
For example:

- `DK` corresponds to Denmark at country level
- `DE_BW` could represent a subnational unit
- more detailed keys may represent local areas

To avoid double-counting, we keep only rows where `location_key` contains no underscore.  
We apply the same logic to the government response dataset.

In [8]:
epi_country = epi[epi["location_key"].astype(str).str.count("_") == 0].copy()
gov_country = gov[gov["location_key"].astype(str).str.count("_") == 0].copy()

print("Country-level epidemiology shape:", epi_country.shape)
print("Country-level government response shape:", gov_country.shape)

print("Unique country-level locations in epidemiology:", epi_country["location_key"].nunique())
print("Unique country-level locations in government response:", gov_country["location_key"].nunique())

Country-level epidemiology shape: (227879, 10)
Country-level government response shape: (172116, 22)
Unique country-level locations in epidemiology: 232
Unique country-level locations in government response: 186


## 2. Time parsing

The `date` column is parsed as a datetime variable in both datasets.  
This is necessary for:

- filtering by year
- aggregating by month
- rolling averages
- and time-series plotting

In [9]:
print(epi_country["date"].dtype)
print(gov_country["date"].dtype)

datetime64[ns]
datetime64[ns]


## 3. Negative daily values

Some daily epidemiological variables contain negative values.  
These are not literal negative cases or deaths, but usually reflect retrospective reporting corrections.

For the visual analysis, we clip negative daily values to zero in order to produce cleaner and more interpretable time series.

In [10]:
for col in ["new_confirmed", "new_deceased", "new_recovered", "new_tested"]:
    if col in epi_country.columns:
        epi_country[col] = epi_country[col].clip(lower=0)

## 4. Rolling averages

Daily COVID-19 data is noisy because of weekend effects, reporting delays, and corrections.  
To make broad pandemic waves more visible, we use 7-day rolling averages for the main time-series plots.

In [11]:
epi_country = epi_country.sort_values(["location_key", "date"]).copy()
gov_country = gov_country.sort_values(["location_key", "date"]).copy()

epi_country["new_confirmed_7d"] = (
    epi_country.groupby("location_key")["new_confirmed"]
    .transform(lambda s: s.rolling(7, min_periods=1).mean())
)

epi_country["new_deceased_7d"] = (
    epi_country.groupby("location_key")["new_deceased"]
    .transform(lambda s: s.rolling(7, min_periods=1).mean())
)

gov_country["stringency_7d"] = (
    gov_country.groupby("location_key")["stringency_index"]
    .transform(lambda s: s.rolling(7, min_periods=1).mean())
)

## 5. Population adjustment

For direct country comparison, raw case counts and death counts are misleading because countries have very different population sizes.  
To make Denmark, Germany, and Great Britain comparable, we normalize selected variables by population and report values such as:

- cases per 100,000 people
- deaths per 100,000 people

This makes the comparison more interpretable and more statistically fair.

In [12]:
population_map = {
    "DK": 5900000,    # Denmark
    "DE": 84400000,   # Germany
    "GB": 67700000,   # Great Britain / United Kingdom
}

## 6. Country-code conversion for maps

For choropleth maps, Plotly expects ISO-3 country codes, while the dataset uses ISO-2 style country identifiers.

We therefore convert country codes using `pycountry`, with a small manual exception for Kosovo (`XK`).

In [13]:
special_cases = {
    "XK": "XKX"
}

def iso2_to_iso3(code):
    if pd.isna(code):
        return None
    code = str(code).strip().upper()
    if code in special_cases:
        return special_cases[code]
    country = pycountry.countries.get(alpha_2=code)
    return country.alpha_3 if country else None

## 7. Creating the cleaned merged dataset

After filtering and preparing the two datasets, we merge them at country level using the shared keys:

- `location_key`
- `date`

This merged dataset makes it possible to compare epidemiological measures and government response measures directly in the same time series.

In [15]:
merged_clean = pd.merge(
    epi_country,
    gov_country,
    on=["location_key", "date"],
    how="inner",
    suffixes=("_epi", "_gov")
)

print("Merged cleaned dataset shape:", merged_clean.shape)
print(merged_clean.head())

Merged cleaned dataset shape: (171055, 33)
        date location_key  new_confirmed  new_deceased  new_recovered  \
0 2020-01-01           AD            0.0           0.0            NaN   
1 2020-01-02           AD            0.0           0.0            NaN   
2 2020-01-03           AD            0.0           0.0            NaN   
3 2020-01-04           AD            0.0           0.0            NaN   
4 2020-01-05           AD            0.0           0.0            NaN   

   new_tested  cumulative_confirmed  cumulative_deceased  \
0         NaN                   0.0                  0.0   
1         NaN                   0.0                  0.0   
2         NaN                   0.0                  0.0   
3         NaN                   0.0                  0.0   
4         NaN                   0.0                  0.0   

   cumulative_recovered  cumulative_tested  ...  international_support  \
0                   NaN                NaN  ...                    0.0   
1        

## Using the final cleaned merged file

In addition to the raw datasets, our project also includes a cleaned merged dataset stored in the `data` folder. This file reflects the final preprocessing work used in the analysis.

For the main analysis sections of this notebook, we can load the cleaned merged file directly. We still document the preprocessing logic above so that the workflow remains transparent and reproducible.

In [17]:
cleaned_path = current_dir.parent / "Data" / "merged_covid_data.csv"

merged_clean = pd.read_csv(cleaned_path, parse_dates=["date"], low_memory=False)

print("Loaded cleaned merged file:", merged_clean.shape)
merged_clean.head()

Loaded cleaned merged file: (170140, 30)


,date,location_key,new_confirmed,new_deceased,new_recovered,new_tested,cumulative_confirmed,cumulative_deceased,cumulative_recovered,cumulative_tested,...,fiscal_measures,international_support,public_information_campaigns,testing_policy,contact_tracing,emergency_investment_in_healthcare,investment_in_vaccines,facial_coverings,vaccination_policy,stringency_index
0,2020-01-01,AD,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2020-01-02,AD,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2020-01-03,AD,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2020-01-04,AD,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2020-01-05,AD,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# ________________________________________________________________
# Exploratory Data Analysis

The exploratory analysis had two purposes:

1. to understand the global structure of the data
2. to identify an analytically meaningful subset of countries for closer comparison

## Global patterns

The first global plots showed the broad development of:

- new confirmed cases over time
- new deaths over time
- average government stringency over time

These plots revealed clear global waves and suggested that the timing of infections, deaths, and restrictions differed substantially across pandemic phases.

## Country comparison

We initially considered a broader global comparison, but eventually narrowed the focus to Denmark, Germany, and Great Britain. This narrower comparison produced a more coherent story because the countries are more regionally and institutionally comparable.

## Key exploratory insight

A central insight from the early analysis was that **official case counts should be interpreted carefully**. Confirmed cases reflect not only underlying spread, but also testing, reporting, and detection practices. This motivated our later emphasis on comparing cases, deaths, and government stringency together rather than interpreting confirmed cases alone.

# Data Analysis

Our analysis combines descriptive comparison and visual interpretation.

## Main analytical questions

We focus on the following questions:

1. How did confirmed cases develop over time in Denmark, Germany, and Great Britain?
2. How did death patterns differ across the three countries?
3. How did government stringency vary over time across the three countries?
4. Is there a visible association between stronger government stringency and later changes in spread or mortality?
5. To what extent might official case numbers underrepresent the actual spread of the pandemic?

## Analytical strategy

We do not attempt to make a strong causal claim. Instead, we use comparative time-series analysis to examine whether:

- waves occurred at similar or different times,
- stringency responses were earlier or later,
- stricter periods appear associated with later changes in cases or deaths,
- and whether deaths provide a different impression of pandemic severity than confirmed cases alone.

## Population-adjusted comparison

To avoid misleading comparisons due to different country sizes, we use per-capita measures in the later country comparison figures. This is especially important when comparing Denmark with larger countries such as Germany and Great Britain.

## Machine learning

We do not use machine learning in this project. The analysis is based on descriptive statistics, time-series comparison, aggregation, and visual narrative design.

# Genre and Narrative Design

## Which genre of data story did we use?

Our project uses a **martini-glass structure**. The story begins with a guided, author-driven introduction to the global pandemic and the country comparison, and then moves into a more exploratory mode where readers can inspect specific countries, time periods, and variables.

This structure is appropriate because the topic is complex and benefits from initial guidance, but also contains enough richness to support interactive exploration.

## Which tools did we use from the categories of Visual Narrative?

We use several visual narrative tools inspired by Segel and Heer:

### Visual structuring
- a consistent color palette
- repeated country colors across plots
- a consistent dark background and layout style
- repeated plot types for easier comparison

### Highlighting
- color coding for countries
- filled line charts for emphasis
- red policy overlay lines for government response
- faceting and small multiples to isolate country trajectories clearly

### Transition guidance
- a progression from global to country-level views
- repeated time axes and plotting conventions
- interactive controls such as dropdowns and range sliders to support guided exploration

## Which tools did we use from the categories of Narrative Structure?

### Ordering
We use a guided sequence:
1. global overview
2. country comparison
3. policy comparison
4. per-capita comparison
5. monthly heatmaps and interactive exploration

### Interactivity
We use:
- Plotly hover interaction
- zooming
- range sliders
- year dropdowns
- metric switching

These features allow users to explore the same story from different temporal perspectives.

### Messaging
We use:
- titles
- explanatory subtitles
- consistent labeling
- and surrounding website text

The website carries the main narrative, while the notebook explains the analytical and design logic behind it.

# Visualizations

## Why these visualizations?

We selected visualizations that match both the structure of the data and the story we want to tell.

## 1. Global time-series plots

These plots show:
- global confirmed cases
- global deaths
- average global stringency

They are useful for establishing the broad temporal structure of the pandemic and introducing the reader to major global waves.

## 2. Choropleth world maps

The world maps show total infections and deaths by country. These provide a global overview of the geographical burden of the pandemic.

## 3. Country comparison line charts

These plots compare Denmark, Germany, and Great Britain over time. They are well suited for showing:
- wave timing
- relative magnitude
- differences in policy response

## 4. Population-adjusted small multiples

These are especially important because they allow fairer comparison between countries of different sizes. Shared axes across panels make comparisons more honest and less visually distorted.

## 5. Interactive Plotly charts

These allow users to:
- zoom into periods of interest
- compare years
- switch metrics
- and inspect exact values

This interactivity supports deeper exploration without overwhelming the main website narrative.

## 6. Monthly heatmaps

The heatmaps summarize long time series into a compact structure. They are particularly useful for comparing:
- waves across months
- differences between years
- and changes in government stringency over time

# Discussion

## What went well?

Several aspects of the project worked well:

- The combination of epidemiology and policy-response data allowed us to tell a richer story than either dataset alone.
- Filtering to country level prevented double-counting and made the analysis more valid.
- Population-adjusted figures greatly improved fairness in country comparison.
- The combination of static and interactive visualizations made it possible to support both storytelling and exploration.

## What is still missing?

There are also important limitations:

- Confirmed case numbers likely underrepresent the true spread of COVID-19, especially in earlier phases of the pandemic.
- Stringency indices summarize policy, but they do not directly measure compliance or enforcement.
- The analysis is descriptive and cannot establish causality.
- Population values were added externally and should be carefully documented and sourced in the final version.

## What could be improved, and why?

Possible improvements include:

- adding testing data where reliable
- including vaccination timing more explicitly
- using lagged comparisons between stringency and later cases/deaths
- adding per-capita maps
- and improving the website’s explanatory annotations around major pandemic phases

These improvements would strengthen both the analytical depth and the interpretability of the story.

# Contributions

This section should briefly describe which group member was primarily responsible for which part of the work.

Example structure:

- **Member 1**: led data cleaning and preprocessing, including country-level filtering and time-series preparation.
- **Member 2**: led exploratory analysis and comparison of epidemiology indicators across countries.
- **Member 3**: led interactive visualization development and Plotly implementation.
- **Member 4**: led narrative design, website text, and integration of Segel and Heer's framework.

All group members contributed to discussion, interpretation, and revision of the final project, but each member had a primary area of responsibility as listed above.

# References

This section should contain all sources used in the project, including:

- dataset sources
- academic references
- visual narrative references
- and any external population data sources

Example entries:

- Google COVID-19 Open Data. Epidemiology dataset.
- Google COVID-19 Open Data. Oxford Government Response dataset.
- Segel, E., & Heer, J. *Narrative Visualization: Telling Stories with Data*.
- Any population source used for per-capita normalization.